# ga-pp-testset-eval

**CPU only · Internet ON · no training · attached subset optional.**

Run the eight numbered code cells in order. If the Kaggle secret **HF_TOKEN** exists,
it is used for Hugging Face downloads and HTTP Range Authorization headers. Otherwise
downloads use anonymous access. Credentials are never printed or saved.

Cell 2 retains its test-image range-fetch behavior (with optional authentication).
Cell 3 downloads the two annotation ZIPs into `/tmp/ga_eval`, extracts official test
annotations, and **retains the ZIPs for cell 4**. Both ZIPs are deleted only after
cell 4 finishes successfully. Cached completed packs skip both ZIP and prefix downloads.

Cell 4 makes **no access to the attached input dataset**. It reads the image archive's
inclusive range `bytes=0-3000000000`, requires HTTP 206 and an exact Content-Range, and
uses stream-unzip local-header parsing. Complete JPEGs go directly into the blob;
the sole accepted error is final TruncatedDataError at the intentional range boundary.
Any partial final JPEG is rolled back. Streamed scenes define SUBSET_SCENES; its
expected count is 46,006 (a differing count warns and continues).

Local test JPEGs are then appended without duplicate scenes. With the expected data,
33 official test scenes are already in the range and 587 additional JPEGs are fetched
by cell 2. If the attachment is absent, cell 2 may fetch all test scenes; cell 4 skips
the scenes already streamed. Every JPEG is checked against ZIP CRC32 and size metadata.

Two sequential LOCAL annotation ZIP passes retain scenes in SUBSET_SCENES or object keys
in the official test lists. Instructions are single-string pickle payloads; labels
are CPU float32 tensors (N,6). Labels are processed in ZIP member order and paired by
identical stems with the in-memory prompt dictionary. No per-file threads or network
mount sample reads are used by cell 4.

Outputs stay under `/kaggle/working/ga_eval/` with a **6 GB** limit. The packed NPY,
Parquet schemas, image blob, relative manifests, exclusions, seed-0 scene validation
split, eval_utils.py, packed_io.py and cells 5–8 retain their existing contracts.
Stages are cached independently. Timing targets are <180s for the image stream and
<600s for annotation passes; actual wall times depend on the Kaggle environment and
appear in the unchanged final REPORT.

The geometry uses LGD's flipped-angle image convention with strict IoU >0.25 and
angle difference <30°. Any failed G1 check is preserved in the REPORT before its
final assertion raises. G1 is a sanity check, not a model evaluation score.

Sources: [LGD geometry](https://github.com/Fsoft-AIC/LGD/blob/main/utils/dataset_processing/grasp.py),
[official test lists](https://github.com/Fsoft-AIC/LGD/tree/main/split/grasp-anything%2B%2B/test),
[base images](https://huggingface.co/datasets/airvlab/Grasp-Anything),
[annotations](https://huggingface.co/datasets/airvlab/Grasp-Anything-pp).


## Cell 1 — Official LGD test split

Setup, CPU-only imports, regex-only split parsing, count assertions and recorded `git ls-remote` commit.


In [ ]:
# CELL 1 — CPU setup and official LGD test keys (the split files are NEVER unpickled).
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['HF_HOME'] = '/tmp/ga_eval/hf'
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['HF_HUB_DISABLE_XET'] = '1'
import sys, subprocess, importlib.util
requirements = {'requests': 'requests', 'tqdm': 'tqdm', 'pandas': 'pandas', 'pyarrow': 'pyarrow',
                'huggingface_hub': 'huggingface_hub', 'shapely': 'shapely', 'skimage': 'scikit-image',
                'matplotlib': 'matplotlib', 'PIL': 'Pillow'}
missing_packages = [package for name, package in requirements.items() if importlib.util.find_spec(name) is None]
if missing_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *missing_packages], check=True)
import io, re, json, gzip, pickle, string, hashlib, shutil, struct, zlib, time, warnings, threading
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
import requests, numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    optional_secret = UserSecretsClient().get_secret('HF_TOKEN')
    HF_TOKEN = optional_secret.strip() if isinstance(optional_secret, str) and optional_secret.strip() else None
    del optional_secret
except Exception as secret_error:
    # Optional secret lookup: report only the exception TYPE, never its message or any credential.
    print('HF_TOKEN unavailable; using anonymous access ('+type(secret_error).__name__+').')
if HF_TOKEN:
    print('Optional Hugging Face secret loaded; authenticated downloads enabled (token hidden).')
else:
    print('Hugging Face downloads will use anonymous access.')

INPUT = Path('/kaggle/input/grasp-anything-pp-subset/ga_subset')
OUT = Path('/kaggle/working/ga_eval')
TMP = Path('/tmp/ga_eval')
LIMIT = 6_000_000_000  # Packed outputs need about 3.5 GB; includes temporary files under OUT.
for path in (OUT, TMP, OUT / 'test_images', OUT / 'splits'):
    path.mkdir(parents=True, exist_ok=True)
CHECK_ATTACHED_INPUT = False  # Optional metadata check; cell 4 never uses the attachment.
if CHECK_ATTACHED_INPUT:
    absent = [name for name in ('images', 'grasp_instructions', 'grasp_label_positive')
              if not (INPUT / name).is_dir()]
    if absent:
        warnings.warn(f'Optional attached subset is absent/incomplete ({absent}); continuing with HTTP/local ZIP sources.')
else:
    print('Attached subset check skipped: the attachment is optional for streamed preparation.')
STEM_RE = re.compile(r'^([0-9a-f]{64})_(\d+)_(\d+)$')
KEY_RE = rb'([0-9a-f]{64}_\d+)'
CODE_VERSION = 'ga-pp-testset-eval-v1'

def digest_json(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        for chunk in iter(lambda: file.read(1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()

def directory_bytes(path):
    return sum(p.stat().st_size for p in Path(path).rglob('*') if p.is_file())

def reserve_output(additional=0):
    used = directory_bytes(OUT)
    if used + additional >= LIMIT:
        raise RuntimeError(f'Output would exceed 6 GB: current={used:,}, additional={additional:,}.')
    if shutil.disk_usage(OUT).free < additional + 100_000_000:
        raise RuntimeError('Insufficient free disk for the next output write.')

def atomic_json(path, value):
    path = Path(path)
    text = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)
    reserve_output(len(text.encode('utf-8')) if path.is_relative_to(OUT) else 0)
    pending = path.with_name(path.name + '.partial')
    pending.write_text(text, encoding='utf-8')
    pending.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def crc_file(path):
    crc = 0
    with Path(path).open('rb') as file:
        for chunk in iter(lambda: file.read(1024**2), b''):
            crc = zlib.crc32(chunk, crc)
    return crc & 0xffffffff

def split_parts(stem):
    match = STEM_RE.fullmatch(stem)
    if match is None:
        raise ValueError(f'Unexpected annotation stem: {stem}')
    scene, obj, part = match.groups()
    return scene, obj, part, scene + '_' + obj

split_meta_path = OUT / 'splits' / 'metadata.json'
if split_meta_path.is_file():
    split_meta = read_json(split_meta_path)
    LGD_COMMIT = split_meta['lgd_commit']
else:
    remote = subprocess.check_output(['git', 'ls-remote', 'https://github.com/Fsoft-AIC/LGD', 'HEAD'], text=True)
    LGD_COMMIT = remote.split()[0]
    if not re.fullmatch(r'[0-9a-f]{40}', LGD_COMMIT):
        raise RuntimeError('git ls-remote did not return an LGD HEAD commit.')
    split_meta = {'lgd_commit': LGD_COMMIT, 'files': {}}

test_keys, test_scenes = {}, {}
EXPECTED_SPLITS = {'seen': (573, 420), 'unseen': (230, 200)}
for split, expected in EXPECTED_SPLITS.items():
    path = OUT / 'splits' / (split + '.obj')
    if not path.is_file():
        url = f'https://raw.githubusercontent.com/Fsoft-AIC/LGD/main/split/grasp-anything%2B%2B/test/{split}.obj'
        with requests.get(url, timeout=(30, 120)) as response:
            response.raise_for_status()
            reserve_output(len(response.content))
            pending = path.with_suffix('.partial')
            pending.write_bytes(response.content)
            pending.replace(path)
    raw = path.read_bytes()
    test_keys[split] = {entry.decode('ascii') for entry in re.findall(KEY_RE, raw)}
    test_scenes[split] = {key[:64] for key in test_keys[split]}
    observed = len(test_keys[split]), len(test_scenes[split])
    print(f'{split}: {observed[0]} unique keys / {observed[1]} scenes; expected {expected[0]} / {expected[1]}')
    if observed != expected:
        warnings.warn(f'Official {split} split differs from the expected counts: {observed} != {expected}')
    assert observed == expected, f'{split} split changed; inspect it before continuing.'
    sha = file_sha(path)
    old = split_meta['files'].get(split)
    if old and old['sha256'] != sha:
        raise RuntimeError(f'Cached {split} split changed after its LGD commit was recorded.')
    split_meta['files'][split] = {'keys': observed[0], 'scenes': observed[1], 'sha256': sha}
assert not (test_keys['seen'] & test_keys['unseen']), 'Seen and unseen object keys overlap.'
ALL_TEST_KEYS = test_keys['seen'] | test_keys['unseen']
ALL_TEST_SCENES = test_scenes['seen'] | test_scenes['unseen']
SPLIT_ID = digest_json({name: sorted(keys) for name, keys in test_keys.items()})
atomic_json(split_meta_path, split_meta)
print(f'Cell 1 complete: CPU only; optional HF authentication; LGD HEAD={LGD_COMMIT}; union={len(ALL_TEST_SCENES)} scenes.')


## Cell 2 — Range-fetch test images

Verify ZIP64 metadata; parse/cache the central directory; fetch missing test JPEGs with eight workers, CRC checks and three attempts with backoff. Protocol/range violations abort immediately.


In [ ]:
# CELL 2 — ZIP64 central directory + independent HTTP ranges for test JPEGs only.
A_URL = 'https://huggingface.co/datasets/airvlab/Grasp-Anything/resolve/main/image_part_aa'
B_URL = 'https://huggingface.co/datasets/airvlab/Grasp-Anything/resolve/main/image_part_ab'
A_SIZE, B_SIZE = 34_359_738_368, 30_653_099_134
TOTAL_SIZE = A_SIZE + B_SIZE
EXPECTED_CD = {'offset': 64_858_429_428, 'size': 154_407_976, 'entries': 994_861}
ARCHIVE_ID = digest_json([A_URL, B_URL, A_SIZE, B_SIZE, EXPECTED_CD])
_sessions = threading.local()

class RangeSafetyError(RuntimeError):
    pass

class DataIntegrityError(IOError):
    pass

def http_session():
    if not hasattr(_sessions, 'session'):
        _sessions.session = requests.Session()
    return _sessions.session

def retry_io(function, label):
    for attempt in range(3):
        try:
            return function()
        except (requests.RequestException, OSError) as error:
            if attempt == 2:
                raise
            delay = 2**attempt
            tqdm.write(f'Retry {attempt+2}/3: {label}: {type(error).__name__}; backoff={delay}s')
            time.sleep(delay)

def _part_range(url, start, size, part_size):
    end = start + size - 1
    headers = {'Range': f'bytes={start}-{end}', 'Accept-Encoding': 'identity'}
    if HF_TOKEN:
        headers['Authorization'] = 'Bearer '+HF_TOKEN
    with http_session().get(url, headers=headers, stream=True, allow_redirects=True,
                            timeout=(30, 180)) as response:
        # Range/protocol failures abort immediately; they are NOT retryable I/O failures.
        if response.status_code != 206:
            raise RangeSafetyError(f'ABORT: HTTP {response.status_code}, expected 206; body not read.')
        expected = f'bytes {start}-{end}/{part_size}'
        if response.headers.get('Content-Range') != expected:
            raise RangeSafetyError(f'ABORT: wrong Content-Range; expected {expected!r}; body not read.')
        if response.headers.get('Content-Encoding', 'identity').lower() != 'identity':
            raise RangeSafetyError('ABORT: encoded response cannot preserve ZIP byte offsets.')
        length = response.headers.get('Content-Length')
        if length is not None and int(length) != size:
            raise RangeSafetyError('ABORT: Content-Length does not equal requested range size.')
        result = bytearray()
        while len(result) < size:
            chunk = response.raw.read(min(1024**2, size-len(result)), decode_content=False)
            if not chunk:
                raise DataIntegrityError(f'Incomplete HTTP range: {len(result):,}/{size:,} bytes.')
            if len(result) + len(chunk) > size:
                raise RangeSafetyError('ABORT: range source exceeded its byte cap.')
            result.extend(chunk)
        return bytes(result)  # No unbounded response.content or full-archive request.

def read_range(o, n):
    """Read exactly n bytes at a combined ZIP offset, splitting an A/B boundary crossing."""
    if not isinstance(o, int) or not isinstance(n, int) or o < 0 or n < 0 or o+n > TOTAL_SIZE:
        raise ValueError(f'Invalid combined range: offset={o}, size={n}.')
    chunks = []
    while n:
        in_a = o < A_SIZE
        local = o if in_a else o-A_SIZE
        part_size = A_SIZE if in_a else B_SIZE
        take = min(n, part_size-local)
        chunks.append(_part_range(A_URL if in_a else B_URL, local, take, part_size))
        o, n = o+take, n-take
    return b''.join(chunks)

def parse_zip64_footer(tail, tail_start):
    locator = tail.rfind(b'PK\x06\x07')
    if locator < 0 or locator + 20 > len(tail):
        raise DataIntegrityError('ZIP64 EOCD locator (0x07064b50) not found in the B tail.')
    sig, disk, record_offset, disks = struct.unpack_from('<4sIQI', tail, locator)
    pos = record_offset-tail_start
    if disk != 0 or disks != 1 or pos < 0 or pos+56 > len(tail):
        raise DataIntegrityError('Unexpected ZIP64 disk layout or EOCD offset.')
    if tail[pos:pos+4] != b'PK\x06\x06':
        raise DataIntegrityError('ZIP64 EOCD record (0x06064b50) does not match its locator.')
    sig, record_size, made, needed, disk, cd_disk, disk_entries, entries, size, offset = \
        struct.unpack_from('<4sQ2H2I4Q', tail, pos)
    if record_size < 44 or pos+12+record_size != locator or disk or cd_disk or disk_entries != entries:
        raise DataIntegrityError('Inconsistent ZIP64 EOCD fields.')
    end_pos = locator + 20
    if tail[end_pos:end_pos+4] != b'PK\x05\x06' or end_pos+22 > len(tail):
        raise DataIntegrityError('Classic EOCD missing after ZIP64 locator.')
    comment_size = struct.unpack_from('<H', tail, end_pos+20)[0]
    if end_pos+22+comment_size != len(tail) or offset+size > record_offset:
        raise DataIntegrityError('EOCD comment length or central-directory extent is inconsistent.')
    return {'offset': offset, 'size': size, 'entries': entries}

def zip64_sizes(extra, usize, csize, offset, disk):
    required = (usize == 0xffffffff, csize == 0xffffffff, offset == 0xffffffff, disk == 0xffff)
    if not any(required):
        return usize, csize, offset, disk
    cursor, payload = 0, None
    while cursor+4 <= len(extra):
        tag, size = struct.unpack_from('<HH', extra, cursor)
        cursor += 4
        if cursor+size > len(extra):
            raise DataIntegrityError('Truncated ZIP extra field.')
        if tag == 0x0001:
            payload = extra[cursor:cursor+size]
            break
        cursor += size
    if payload is None:
        raise DataIntegrityError('ZIP64 sentinel without ZIP64 extra field 0x0001.')
    values, cursor = [usize, csize, offset, disk], 0
    for index, needed in enumerate(required):
        if needed:
            width, fmt = (4, '<I') if index == 3 else (8, '<Q')
            if cursor+width > len(payload):
                raise DataIntegrityError('ZIP64 extra lacks a required field.')
            values[index] = struct.unpack_from(fmt, payload, cursor)[0]
            cursor += width
    return tuple(values)

def parse_central_directory(data, expected_entries):
    records, position, entries = {}, 0, 0
    while position < len(data):
        if position+46 > len(data) or data[position:position+4] != b'PK\x01\x02':
            raise DataIntegrityError(f'Invalid central file header at directory byte {position:,}.')
        fields = struct.unpack_from('<4s6H3I5H2I', data, position)
        flags, method, crc, csize, usize = fields[3], fields[4], fields[7], fields[8], fields[9]
        name_len, extra_len, comment_len, disk, offset = fields[10], fields[11], fields[12], fields[13], fields[16]
        end = position+46+name_len+extra_len+comment_len
        if end > len(data):
            raise DataIntegrityError('Truncated central-directory member.')
        start = position+46
        name = data[start:start+name_len].decode('utf-8' if flags & 0x800 else 'cp437')
        extra = data[start+name_len:start+name_len+extra_len]
        usize, csize, offset, disk = zip64_sizes(extra, usize, csize, offset, disk)
        if disk != 0 or offset+csize > TOTAL_SIZE:
            raise DataIntegrityError('Unexpected central-directory disk/offset.')
        basename = name.replace('\\', '/').rsplit('/', 1)[-1]
        if basename:
            if basename in records:
                raise DataIntegrityError(f'Duplicate ZIP basename: {basename}')
            records[basename] = {'name': name, 'method': method, 'flags': flags, 'crc32': crc,
                                'comp_size': csize, 'uncomp_size': usize, 'local_offset': offset}
        entries += 1
        position = end
    if entries != expected_entries:
        raise DataIntegrityError(f'Expected {expected_entries:,} CD entries; parsed {entries:,}.')
    return records

index_path, index_meta_path = OUT / 'cd_index.parquet', OUT / 'cd_index.meta.json'
cache_ok = index_path.is_file() and index_meta_path.is_file()
if cache_ok:
    index_meta = read_json(index_meta_path)
    cache_ok = index_meta['archive_id'] == ARCHIVE_ID and index_meta['parquet_sha256'] == file_sha(index_path)
if cache_ok:
    cd_index = pd.read_parquet(index_path).set_index('basename').to_dict('index')
    print(f'Skip CD HTTP requests: verified cached index with {len(cd_index):,} basenames.')
else:
    tail = retry_io(lambda: read_range(TOTAL_SIZE-200_000, 200_000), 'ZIP64 B tail')
    cd_metadata = parse_zip64_footer(tail, TOTAL_SIZE-200_000)
    if cd_metadata != EXPECTED_CD:
        raise DataIntegrityError(f'ZIP64 metadata differs from supplied constants: {cd_metadata}')
    print('Verified ZIP64 EOCD + locator:', cd_metadata)
    # This entire CD is inside B, so read_range performs ONE ranged GET (~154 MB).
    data = retry_io(lambda: read_range(cd_metadata['offset'], cd_metadata['size']), 'central directory')
    cd_index = parse_central_directory(data, cd_metadata['entries'])
    del data, tail
    frame = pd.DataFrame.from_dict(cd_index, orient='index').rename_axis('basename').reset_index()
    # Reserve uncompressed DataFrame bytes conservatively; the final Parquet uses zstd.
    reserve_output(int(frame.memory_usage(index=True, deep=True).sum()))
    pending = index_path.with_suffix('.parquet.partial')
    frame.to_parquet(pending, index=False, compression='zstd')
    pending.replace(index_path)
    del frame
    index_meta = {'archive_id': ARCHIVE_ID, 'zip64': cd_metadata, 'indexed_basenames': len(cd_index),
                  'parquet_sha256': file_sha(index_path)}
    atomic_json(index_meta_path, index_meta)

def fetch_image(scene):
    record = cd_index[scene + '.jpg']
    target = OUT / 'test_images' / (scene + '.jpg')
    expected_size, expected_crc = int(record['uncomp_size']), int(record['crc32'])
    if not 0 < expected_size <= 64*1024**2 or not 0 < int(record['comp_size']) <= 64*1024**2:
        raise DataIntegrityError(f'Unexpectedly large/empty JPEG record: {scene}.jpg')
    if target.is_file() and target.stat().st_size == expected_size and crc_file(target) == expected_crc:
        return 'cached'
    def attempt():
        offset, csize = int(record['local_offset']), int(record['comp_size'])
        header = read_range(offset, 30)
        if header[:4] != b'PK\x03\x04':
            raise DataIntegrityError(f'Bad local header for {scene}.jpg')
        sig, version, flags, method, tm, dt, crc, local_comp, local_uncomp, name_len, extra_len = \
            struct.unpack('<4s5H3I2H', header)
        if method != int(record['method']) or flags != int(record['flags']) or flags & 1:
            raise DataIntegrityError(f'Unexpected local flags/method/encryption for {scene}.jpg')
        # LOCAL header lengths, never central-directory filename/extra lengths.
        payload = read_range(offset+30+name_len+extra_len, csize)
        if method == 0:
            image = payload
        elif method == 8:
            inflater = zlib.decompressobj(-15)
            try:
                image = inflater.decompress(payload, expected_size+1)
                if len(image) > expected_size or inflater.unconsumed_tail:
                    raise DataIntegrityError(f'Deflate output exceeds declared JPEG size for {scene}.jpg')
                image += inflater.flush()
            except zlib.error as error:
                raise DataIntegrityError(f'Deflate failed for {scene}.jpg') from error
            if not inflater.eof or inflater.unused_data or inflater.unconsumed_tail:
                raise DataIntegrityError(f'Incomplete/trailing Deflate data for {scene}.jpg')
        else:
            raise DataIntegrityError(f'Unsupported ZIP method {method} for {scene}.jpg')
        if len(image) != expected_size or (zlib.crc32(image) & 0xffffffff) != expected_crc:
            raise DataIntegrityError(f'Size/CRC32 mismatch for {scene}.jpg')
        pending = target.with_suffix('.jpg.partial')
        pending.write_bytes(image)
        pending.replace(target)
        return 'downloaded'
    return retry_io(attempt, scene[:12] + '.jpg')

already_input = {scene for scene in ALL_TEST_SCENES if (INPUT / 'images' / (scene+'.jpg')).is_file()}
needed = ALL_TEST_SCENES-already_input
found = {scene for scene in needed if scene+'.jpg' in cd_index}
missing = needed-found
reserve_output(sum(int(cd_index[scene+'.jpg']['uncomp_size']) for scene in found))
fetch_counts = Counter()
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(fetch_image, scene): scene for scene in sorted(found)}
    try:
        for future in tqdm(as_completed(futures), total=len(futures), desc='Test JPEG ranges', unit='images'):
            fetch_counts[future.result()] += 1  # Any exhausted retry or safety error propagates.
    except BaseException:
        for pending in futures:
            pending.cancel()  # Abort queued work; at most eight already-running bounded fetches finish.
        raise
image_fetch = {'required_scenes': len(ALL_TEST_SCENES), 'already_in_subset': len(already_input),
               'found_in_zip': len(found), 'missing_from_zip': len(missing),
               'downloaded_this_run': fetch_counts['downloaded'], 'cached_output': fetch_counts['cached'],
               'ready': len(already_input)+len(found), 'missing_scene_ids': sorted(missing)}
atomic_json(OUT / 'image_fetch.json', image_fetch)
print('Cell 2 complete:', image_fetch)
reserve_output()


## Cell 3 — Test annotations; retain source ZIPs

Extract official test-key annotations locally, keeping both ZIPs for cell 4. Optional HF_TOKEN is passed explicitly to Hugging Face downloads.


In [ ]:
# CELL 3 — Official test annotations; retain LOCAL source ZIPs until cell 4 succeeds.

def ensure_annotation_zips():
    """Return valid LOCAL archive paths; use the optional secret, otherwise anonymous."""
    import zipfile
    paths = {}
    for kind in ('grasp_instructions', 'grasp_label_positive'):
        path = TMP / (kind+'.zip')
        if not path.is_file():
            expected_bytes = 1_600_000_000 if kind == 'grasp_instructions' else 4_100_000_000
            if shutil.disk_usage(TMP).free < expected_bytes+100_000_000:
                raise RuntimeError(f'Insufficient /tmp space for {path.name}.')
            downloaded = hf_hub_download(repo_id='airvlab/Grasp-Anything-pp', filename=path.name,
                repo_type='dataset', local_dir=str(TMP), token=HF_TOKEN or False, revision='main')
            if Path(downloaded).resolve() != path.resolve():
                raise RuntimeError(f'Unexpected local ZIP destination: {path.name}')
        if not zipfile.is_zipfile(path):
            raise DataIntegrityError(f'Incomplete local annotation ZIP: {path.name}; remove it and rerun cell 3.')
        paths[kind] = path
    return paths

def stream_packed_ready():
    marker = OUT / 'packed' / 'stream_complete.json'
    if not marker.is_file():
        return False
    saved = read_json(marker)
    if saved.get('source_version') != 'http-prefix-local-zip-v1' or saved.get('split_id') != SPLIT_ID:
        return False
    return all((OUT / 'packed' / name).is_file() and
               (OUT / 'packed' / name).stat().st_size == size for name, size in saved['files'].items())

ANN = OUT / 'test_ann'
ANN_KINDS = ('grasp_instructions', 'grasp_label_positive')
for kind in ANN_KINDS:
    (ANN / kind).mkdir(parents=True, exist_ok=True)
annotation_marker = OUT / 'annotations.json'
annotation_ready = False
if annotation_marker.is_file():
    annotation_meta = read_json(annotation_marker)
    annotation_ready = annotation_meta['split_id'] == SPLIT_ID
    if annotation_ready:
        annotation_ready = all((ANN / kind / name).is_file() and
            (ANN / kind / name).stat().st_size == size and crc_file(ANN / kind / name) == crc
            for kind, records in annotation_meta['files'].items() for name, size, crc in records)
if not annotation_ready or not stream_packed_ready():
    ensure_annotation_zips()
if annotation_ready:
    print('Selected test annotations already pass size/CRC checks; source ZIPs are reserved for cell 4.')
else:
    selected, zip_paths = {}, {}
    for kind in ANN_KINDS:
        filename = kind + '.zip'
        path = TMP / filename
        if not path.is_file():
            expected_bytes = 1_600_000_000 if kind == 'grasp_instructions' else 4_100_000_000
            if shutil.disk_usage(TMP).free < expected_bytes+100_000_000:
                raise RuntimeError(f'Not enough /tmp space for {filename}.')
            path = Path(hf_hub_download(repo_id='airvlab/Grasp-Anything-pp', filename=filename,
                repo_type='dataset', local_dir=str(TMP), token=HF_TOKEN or False, revision='main'))
        zip_paths[kind] = path
        import zipfile
        extension = '.pkl' if kind == 'grasp_instructions' else '.pt'
        chosen, used_basenames = [], set()
        with zipfile.ZipFile(path) as archive:
            for info in tqdm(archive.infolist(), desc=f'Index {kind}', unit='members'):
                if info.is_dir():
                    continue
                basename = PurePosixPath(info.filename.replace('\\', '/')).name
                if not basename.endswith(extension):
                    continue
                match = STEM_RE.fullmatch(Path(basename).stem)
                if match is None or match.group(1)+'_'+match.group(2) not in ALL_TEST_KEYS:
                    continue
                if basename in used_basenames:
                    raise RuntimeError(f'Duplicate selected ZIP basename: {basename}')
                used_basenames.add(basename)
                chosen.append([info.filename, basename, info.file_size, info.CRC])
            valid_names = {name for member, name, size, crc in chosen
                if (ANN / kind / name).is_file() and (ANN / kind / name).stat().st_size == size
                and crc_file(ANN / kind / name) == crc}
            additional = sum(size for member, name, size, crc in chosen if name not in valid_names)
            reserve_output(additional)
            for member, basename, size, crc in tqdm(chosen, desc=f'Extract {kind}', unit='files'):
                target = ANN / kind / basename
                if basename in valid_names:
                    continue
                pending = target.with_suffix(target.suffix + '.partial')
                try:
                    written, actual_crc = 0, 0
                    with archive.open(member) as source, pending.open('wb') as output:
                        for chunk in iter(lambda: source.read(1024**2), b''):
                            output.write(chunk)
                            written += len(chunk)
                            actual_crc = zlib.crc32(chunk, actual_crc)
                    if written != size or (actual_crc & 0xffffffff) != crc:
                        raise DataIntegrityError(f'Annotation size/CRC mismatch: {member}')
                    pending.replace(target)
                finally:
                    if pending.exists():
                        pending.unlink()
        selected[kind] = [[name, size, crc] for member, name, size, crc in chosen]
        print(f'{kind}: selected {len(chosen):,} files; no other members extracted.')
    annotation_meta = {'split_id': SPLIT_ID, 'files': selected}
    atomic_json(annotation_marker, annotation_meta)

prompt_stems = {Path(name).stem for name, size, crc in annotation_meta['files']['grasp_instructions']}
label_stems = {Path(name).stem for name, size, crc in annotation_meta['files']['grasp_label_positive']}
test_pair_stems = prompt_stems & label_stems
annotation_counts = {}
for split, keys in test_keys.items():
    stems = {stem for stem in test_pair_stems if split_parts(stem)[3] in keys}
    represented = {split_parts(stem)[3] for stem in stems}
    annotation_counts[split] = {'pairs': len(stems), 'keys_with_zero_pairs': len(keys-represented),
                              'zero_pair_keys': sorted(keys-represented)}
    print(f'{split}: {len(stems):,} (prompt,label) pairs; {len(keys-represented)} keys with zero pairs.')
annotation_counts['unpaired_prompt_files'] = len(prompt_stems-label_stems)
annotation_counts['unpaired_label_files'] = len(label_stems-prompt_stems)
atomic_json(OUT / 'annotation_counts.json', annotation_counts)
kept_zips = [kind+'.zip' for kind in ANN_KINDS if (TMP / (kind+'.zip')).is_file()]
print('Cell 3 complete:', len(test_pair_stems), 'test annotation pairs; local source ZIPs retained:',
      kept_zips if kept_zips else 'completed streamed pack already cached; no ZIP download needed')
reserve_output()


## Cell 4 — Streamed subset images and sequential local annotation ZIPs

Recreate SUBSET_SCENES from the bounded HTTP prefix, append local test JPEGs, and pack labels/prompts with two sequential ZIP passes. Preserve manifests/splits and print per-stage timings. Delete source ZIPs only on successful completion.


In [ ]:
# CELL 4 — HTTP prefix -> JPEG blob; sequential LOCAL ZIP passes -> packed labels and manifests.
# No attached-dataset paths, file inventories, metadata calls, or sample reads are used here.
import importlib.util
if importlib.util.find_spec('stream_unzip') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
                    'stream-unzip>=0.0.99,<0.1'], check=True)
from stream_unzip import stream_unzip, TruncatedDataError
import zipfile

SOURCE_VERSION = 'http-prefix-local-zip-v1'
FORCE_REPACK = False
SUBSET_RANGE_END = 3_000_000_000  # Inclusive: 3,000,000,001 compressed bytes.
EXPECTED_SUBSET_SCENES = 46_006
PACKED = OUT / 'packed'
PACKED.mkdir(parents=True, exist_ok=True)
cell4_started = time.perf_counter()
stage_timings, stage_reused = {}, {}

def stage_done(name, started, reused=False):
    stage_timings[name] = time.perf_counter()-started
    stage_reused[name] = bool(reused)
    print(f'{name}: {stage_timings[name]:.2f}s' + (' (reused)' if reused else ''))

def packed_disk_bytes():
    return directory_bytes(OUT)  # OUT is local; never visits the attached dataset.

def reserve_packed(additional=0):
    if packed_disk_bytes()+additional >= LIMIT:
        raise RuntimeError('Packed output would exceed the 6 GB budget.')
    if shutil.disk_usage(OUT).free < additional+100_000_000:
        raise RuntimeError('Insufficient free disk for packed outputs.')

def packed_atomic_json(path, value):
    text = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)
    reserve_packed(len(text.encode('utf-8')))
    pending = path.with_name(path.name+'.partial')
    pending.write_text(text, encoding='utf-8')
    pending.replace(path)

def local_write_capacity():
    return min(LIMIT-packed_disk_bytes()-20_000_000, shutil.disk_usage(OUT).free-100_000_000)

def save_parquet_atomic(frame, path):
    reserve_packed(int(frame.memory_usage(index=True, deep=True).sum())+1_000_000)
    pending = path.with_name(path.name+'.partial')
    frame.to_parquet(pending, index=False, compression='zstd')
    pending.replace(path)

def npy_header(rows):
    # A fixed 4096-byte NPY v2 header lets us append raw rows and update the shape
    # in place, without a second full-size label spool/copy.
    header_size = 4096-12
    value = repr({'descr': '<f4', 'fortran_order': False, 'shape': (int(rows), 6)}).encode('ascii')
    if len(value)+1 > header_size:
        raise ValueError('NPY header exceeds its reserved space.')
    return np.lib.format.magic(2, 0)+struct.pack('<I', header_size)+value+b' '*(header_size-len(value)-1)+b'\n'

def warn_subset_count(scenes):
    try:
        assert len(scenes) == EXPECTED_SUBSET_SCENES
    except AssertionError:
        warnings.warn(f'Range subset has {len(scenes):,} scenes; expected {EXPECTED_SUBSET_SCENES:,}. '
                      'Continuing with the scenes actually recovered from the range.')

# Release old cached blob handles/maps before replacing derived files on rerun.
if 'packed_io' in sys.modules:
    sys.modules['packed_io'].close_packed(OUT)
old_map = globals().pop('packed_labels', None)
if isinstance(old_map, np.memmap):
    old_map._mmap.close()

image_started = time.perf_counter()
blob_path, images_path = PACKED / 'images.bin', PACKED / 'images.parquet'
image_marker = PACKED / 'images.meta.json'
IMAGE_KEY = digest_json([SOURCE_VERSION, SPLIT_ID, A_URL, A_SIZE, SUBSET_RANGE_END])
image_reuse = not FORCE_REPACK and image_marker.is_file() and blob_path.is_file() and images_path.is_file()
if image_reuse:
    image_meta = read_json(image_marker)
    image_reuse = (image_meta.get('source_version') == SOURCE_VERSION and image_meta['key'] == IMAGE_KEY
                   and blob_path.stat().st_size == image_meta['blob_bytes']
                   and file_sha(images_path) == image_meta['images_sha256'])
if image_reuse:
    SUBSET_SCENES = set(image_meta['subset_scenes'])
    image_table = pd.read_parquet(images_path)
    written_scenes = set(image_table['scene'])
    if image_table['scene'].duplicated().any() or not (SUBSET_SCENES | ALL_TEST_SCENES) <= written_scenes:
        raise DataIntegrityError('Cached image pack does not cover the subset and all official test scenes.')
    stage_done('image_stream_and_local_append_s', image_started, True)
else:
    for path in (PACKED / 'stream_complete.json', image_marker, blob_path, images_path,
                 blob_path.with_name('images.bin.partial')):
        path.unlink(missing_ok=True)
    expected = SUBSET_RANGE_END+1
    if not 0 < SUBSET_RANGE_END < A_SIZE:
        raise ValueError('The prefix must end within image_part_aa.')
    capacity = local_write_capacity()
    pending = blob_path.with_name('images.bin.partial')
    state = {'received': 0}
    records, written, SUBSET_SCENES = [], {}, set()
    offset = 0
    boundary = None
    current_entry_start = None
    current_scene = None
    headers = {'Range': f'bytes=0-{SUBSET_RANGE_END}', 'Accept-Encoding': 'identity'}
    if HF_TOKEN:
        headers['Authorization'] = 'Bearer '+HF_TOKEN
    try:
        with pending.open('wb') as output:
            with http_session().get(A_URL, headers=headers, stream=True, allow_redirects=True,
                                    timeout=(30, 180)) as response:
                # Validate before consuming even one response-body byte.
                if response.status_code != 206:
                    raise RangeSafetyError(f'ABORT: HTTP {response.status_code}; expected 206. Body not read.')
                expected_range = f'bytes 0-{SUBSET_RANGE_END}/{A_SIZE}'
                if response.headers.get('Content-Range') != expected_range:
                    raise RangeSafetyError(f'ABORT: expected Content-Range {expected_range!r}; body not read.')
                if response.headers.get('Content-Encoding', 'identity').lower() != 'identity':
                    raise RangeSafetyError('ABORT: encoded response invalidates ZIP byte offsets.')
                length = response.headers.get('Content-Length')
                if length is not None and int(length) != expected:
                    raise RangeSafetyError('ABORT: Content-Length does not match the bounded range.')
                with tqdm(total=expected, unit='B', unit_scale=True, desc='Stream subset image prefix') as progress:
                    def chunks():
                        while state['received'] < expected:
                            data = response.raw.read(min(1024**2, expected-state['received']), decode_content=False)
                            if not data:
                                raise DataIntegrityError(f'Network EOF at {state["received"]:,}/{expected:,} bytes; '
                                                         'not an accepted range-boundary truncation.')
                            if len(data) > expected-state['received']:
                                raise RangeSafetyError('ABORT: response exceeded the configured byte cap.')
                            state['received'] += len(data)
                            progress.update(len(data))
                            yield data
                    try:
                        for raw_name, declared_size, data in stream_unzip(chunks()):
                            basename = raw_name.decode('utf-8').replace('\\', '/').rsplit('/', 1)[-1]
                            if not re.fullmatch(r'[0-9a-f]{64}\.jpg', basename):
                                for _ in data:
                                    pass  # Drain directories/non-JPEGs; stream_unzip still validates them.
                                continue
                            scene = basename[:-4]
                            current_scene, current_entry_start = scene, output.tell()
                            duplicate = scene in written
                            size, crc = 0, 0
                            for chunk in data:
                                size += len(chunk)
                                crc = zlib.crc32(chunk, crc)
                                if size > 64*1024**2:
                                    raise DataIntegrityError(f'Unexpectedly large JPEG member: {basename}')
                                if not duplicate:
                                    if output.tell()+len(chunk) > capacity:
                                        raise RuntimeError('Image blob exceeds the 6 GB output/free-disk budget.')
                                    output.write(chunk)
                            # Exhausting the entry above includes stream_unzip's ZIP CRC/size checks.
                            crc &= 0xffffffff
                            record = cd_index.get(basename)
                            if record is None or size != int(record['uncomp_size']) or crc != int(record['crc32']):
                                raise DataIntegrityError(f'JPEG CRC/size differs from the ZIP directory: {basename}')
                            if declared_size is not None and size != declared_size:
                                raise DataIntegrityError(f'JPEG local-header size mismatch: {basename}')
                            if duplicate:
                                if written[scene][1:] != (size, crc):
                                    raise DataIntegrityError(f'Conflicting duplicate JPEG: {basename}')
                            else:
                                written[scene] = (current_entry_start, size, crc)
                                records.append([scene, current_entry_start, size, crc])
                            SUBSET_SCENES.add(scene)
                            current_entry_start = current_scene = None
                            if len(SUBSET_SCENES) % 1000 == 0:
                                progress.set_postfix(images=len(SUBSET_SCENES))
                    except TruncatedDataError:
                        if state['received'] != expected:
                            raise
                        boundary = f'TruncatedDataError at inclusive range end {SUBSET_RANGE_END:,}'
                        if current_entry_start is not None:
                            output.seek(current_entry_start)
                            output.truncate()
                            print('\nDiscarded incomplete final JPEG from blob:', current_scene)
                        else:
                            print('\nRange ended in a ZIP header; no incomplete JPEG was committed.')
                        print('Accepted intentional boundary:', boundary)
                    if state['received'] != expected:
                        raise DataIntegrityError('ZIP parser stopped before consuming the validated range.')
            if not SUBSET_SCENES:
                raise DataIntegrityError('Range contained no complete JPEG scenes.')
            subset_image_count = len(SUBSET_SCENES)
            stage_done('image_http_stream_s', image_started)
            local_started = time.perf_counter()
            with os.scandir(OUT / 'test_images') as entries:
                local_names = sorted(entry.name for entry in entries
                    if re.fullmatch(r'[0-9a-f]{64}\.jpg', entry.name) and entry.name[:-4] in ALL_TEST_SCENES)
            local_appended = local_duplicates = 0
            for basename in tqdm(local_names, desc='Append local test JPEGs', unit='images'):
                scene = basename[:-4]
                if scene in written:
                    local_duplicates += 1
                    continue
                data = (OUT / 'test_images' / basename).read_bytes()  # LOCAL disk only.
                size, crc = len(data), zlib.crc32(data) & 0xffffffff
                record = cd_index.get(basename)
                if record is None or size != int(record['uncomp_size']) or crc != int(record['crc32']):
                    raise DataIntegrityError(f'Local test JPEG CRC/size mismatch: {basename}')
                offset = output.tell()
                if offset+size > capacity:
                    raise RuntimeError('Image blob exceeds the 6 GB output/free-disk budget.')
                output.write(data)
                written[scene] = (offset, size, crc)
                records.append([scene, offset, size, crc])
                local_appended += 1
            missing = ALL_TEST_SCENES-written.keys()
            if missing:
                raise DataIntegrityError(f'{len(missing)} official test images absent from stream/local cache: '
                                         f'{sorted(missing)[:5]}; rerun cell 2 to fetch them.')
            offset = output.tell()
        pending.replace(blob_path)
    finally:
        pending.unlink(missing_ok=True)
    image_table = pd.DataFrame(records, columns=['scene', 'offset', 'nbytes', 'crc32'])
    save_parquet_atomic(image_table.sort_values('scene'), images_path)
    image_meta = {'key': IMAGE_KEY, 'source_version': SOURCE_VERSION, 'subset_scenes': sorted(SUBSET_SCENES),
        'scenes': len(image_table), 'blob_bytes': offset, 'images_sha256': file_sha(images_path),
        'crc_verified': True, 'stream_received': state['received'], 'boundary': boundary,
        'local_images_appended': local_appended, 'local_duplicates_skipped': local_duplicates,
        'test_scenes_in_subset': len(SUBSET_SCENES & ALL_TEST_SCENES)}
    packed_atomic_json(image_marker, image_meta)
    stage_done('image_local_append_and_index_s', local_started)
warn_subset_count(SUBSET_SCENES)
print(f'Images: streamed subset={len(SUBSET_SCENES):,}; test scenes already in range='
      f'{image_meta["test_scenes_in_subset"]:,}; appended local test images={image_meta["local_images_appended"]:,}; '
      f'local duplicates skipped={image_meta["local_duplicates_skipped"]:,}; blob scenes={len(image_table):,}.')

VERBS = sorted(['take hold of', 'get hold of', 'pick up', 'grab hold of', 'grasp', 'grab', 'hold',
    'take', 'lift', 'fetch', 'seize', 'catch', 'clutch', 'give me', 'bring me', 'hand me',
    'pass me', 'get'], key=len, reverse=True)
SEPARATORS = [' by its ', ' at its ', ' on its ', ' by the ', ' at the ', ' on the ',
              ' from its ', ' with its ']
COLUMNS = ['stem', 'scene', 'obj', 'part', 'prompt', 'obj_name', 'label_start', 'label_count', 'split']

def extract_obj_name(prompt):
    import unicodedata
    text = ''.join(' ' if char in string.punctuation or unicodedata.category(char).startswith('P')
                   else char for char in prompt.lower())
    text = ' '.join(text.split())
    for verb in VERBS:
        if text == verb or text.startswith(verb+' '):
            text = text[len(verb):].lstrip()
            break
    while text.split(' ', 1)[0] in ('the', 'a', 'an'):
        text = text.split(' ', 1)[1] if ' ' in text else ''
    positions = [position for sep in SEPARATORS if (position := text.find(sep)) >= 0]
    name = (text[:min(positions)] if positions else text).strip()
    if len(name) > 3 and name.endswith('s') and not name.endswith('ss'):
        name = name[:-1]
    if not name:
        raise ValueError(f'Object-name normalization produced an empty result: {prompt!r}')
    return name, bool(positions)

def member_stem(info, extension):
    if info.is_dir():
        return None
    basename = info.filename.replace('\\', '/').rsplit('/', 1)[-1]
    if not basename.endswith(extension):
        return None
    stem = basename[:-len(extension)]
    match = STEM_RE.fullmatch(stem)
    if match is None:
        return None
    scene, obj, part = match.groups()
    return stem if scene in SUBSET_SCENES or scene+'_'+obj in ALL_TEST_KEYS else None

annotation_started = time.perf_counter()
label_path, table_path = PACKED / 'labels.npy', PACKED / 'annotations.parquet'
ann_marker = PACKED / 'annotations.meta.json'
ANN_KEY = digest_json([SOURCE_VERSION, SPLIT_ID, sorted(SUBSET_SCENES), 'raw-npy-v2'])
ann_reuse = not FORCE_REPACK and ann_marker.is_file() and label_path.is_file() and table_path.is_file()
if ann_reuse:
    ann_meta = read_json(ann_marker)
    ann_reuse = (ann_meta.get('source_version') == SOURCE_VERSION and ann_meta['key'] == ANN_KEY
                 and label_path.stat().st_size == ann_meta['labels_bytes']
                 and file_sha(table_path) == ann_meta['annotations_sha256'])
if ann_reuse:
    packed_annotations = pd.read_parquet(table_path)
    stage_done('annotation_zip_pass_s', annotation_started, True)
else:
    zip_paths = ensure_annotation_zips()  # Defined by cell 3; authenticated when its secret exists.
    for path in (PACKED / 'stream_complete.json', ann_marker, label_path, table_path,
                 label_path.with_name('labels.npy.partial')):
        path.unlink(missing_ok=True)
    prompt_started = time.perf_counter()
    prompts = {}
    subset_prompt_stems = set()
    with zipfile.ZipFile(zip_paths['grasp_instructions']) as archive:
        for info in tqdm(archive.infolist(), desc='Sequential instruction ZIP', unit='members'):
            stem = member_stem(info, '.pkl')
            if stem is None:
                continue
            if stem in prompts:
                raise DataIntegrityError(f'Duplicate instruction stem: {stem}')
            prompt = pickle.loads(archive.read(info))
            assert type(prompt) is str, f'Expected ONE str prompt: {stem}'
            prompts[stem] = prompt
            if stem[:64] in SUBSET_SCENES:
                subset_prompt_stems.add(stem)
    stage_done('instruction_zip_s', prompt_started)
    label_started = time.perf_counter()
    capacity, rows, records = local_write_capacity(), 0, []
    payload_digest = hashlib.sha256()
    subset_label_stems, paired_stems, official_test_stems = set(), set(), set()
    pending = label_path.with_name('labels.npy.partial')
    try:
        with pending.open('wb') as output, zipfile.ZipFile(zip_paths['grasp_label_positive']) as archive:
            output.write(npy_header(0))
            # Iterate in ZIP MEMBER ORDER; never sort members or launch per-file workers.
            for info in tqdm(archive.infolist(), desc='Sequential positive-label ZIP', unit='members'):
                stem = member_stem(info, '.pt')
                if stem is None:
                    continue
                scene, obj, part, key = split_parts(stem)
                if scene in SUBSET_SCENES:
                    subset_label_stems.add(stem)
                if stem not in prompts:
                    continue
                if stem in paired_stems:
                    raise DataIntegrityError(f'Duplicate positive-label stem: {stem}')
                tensor = torch.load(io.BytesIO(archive.read(info)), map_location='cpu', weights_only=True)
                if not isinstance(tensor, torch.Tensor) or tensor.ndim != 2 or tensor.shape[1] != 6:
                    raise ValueError(f'Expected tensor (N,6): {stem}')
                if tensor.dtype != torch.float32:
                    raise TypeError(f'Expected float32 labels: {stem}')
                raw = tensor.detach().cpu().numpy().astype('<f4', copy=False)
                if 4096+(rows+len(raw))*24 > capacity:
                    raise RuntimeError('Packed labels exceed the 6 GB output/free-disk budget.')
                payload = raw.tobytes(order='C')
                output.write(payload)
                payload_digest.update(payload)
                prompt = prompts[stem]
                obj_name, separated = extract_obj_name(prompt)
                source = 'test' if key in ALL_TEST_KEYS else 'subset'
                records.append([stem, scene, obj, part, prompt, obj_name, rows, len(raw), source])
                paired_stems.add(stem)
                if source == 'test':
                    official_test_stems.add(stem)
                rows += len(raw)
            output.seek(0)
            output.write(npy_header(rows))
        pending.replace(label_path)
    finally:
        pending.unlink(missing_ok=True)
    packed_annotations = pd.DataFrame(records, columns=[
        'stem', 'scene', 'obj', 'part', 'prompt', 'obj_name', 'label_start', 'label_count', 'source'])
    packed_annotations = packed_annotations.sort_values('stem').reset_index(drop=True)
    save_parquet_atomic(packed_annotations, table_path)
    paired_subset = {stem for stem in paired_stems if stem[:64] in SUBSET_SCENES}
    ann_meta = {'key': ANN_KEY, 'source_version': SOURCE_VERSION, 'stems': len(records), 'rows': rows,
        'labels_bytes': label_path.stat().st_size, 'annotations_sha256': file_sha(table_path),
        'labels_payload_sha256': payload_digest.hexdigest(),
        'input_unpaired_prompts': len(subset_prompt_stems-paired_subset),
        'input_unpaired_labels': len(subset_label_stems-paired_subset)}
    packed_atomic_json(ann_marker, ann_meta)
    stage_done('label_zip_and_npy_s', label_started)
    stage_done('annotation_zip_pass_s', annotation_started)
    del prompts
input_stems = set(packed_annotations.loc[packed_annotations['scene'].isin(SUBSET_SCENES), 'stem'])
official_test_stems = set(packed_annotations.loc[
    (packed_annotations['scene']+'_'+packed_annotations['obj']).isin(ALL_TEST_KEYS), 'stem'])
print(f'Annotations: {len(packed_annotations):,} unique stems; {ann_meta["rows"]:,} raw rows; '
      f'{len(input_stems):,} subset candidates; {len(official_test_stems):,} official test pairs.')

manifest_started = time.perf_counter()
manifest_paths = {name: OUT / (name+'.csv') for name in ('test_seen', 'test_unseen', 'train', 'val')}
manifest_meta_path = OUT / 'manifests.json'
manifest_key = digest_json([ANN_KEY, ann_meta['annotations_sha256'], 'packed-manifests-v1'])
manifest_reuse = not FORCE_REPACK and manifest_meta_path.is_file() and all(path.is_file() for path in manifest_paths.values())
if manifest_reuse:
    manifest_meta = read_json(manifest_meta_path)
    manifest_reuse = manifest_meta['key'] == manifest_key and all(
        file_sha(path) == manifest_meta['csv_sha256'][name] for name, path in manifest_paths.items())

def save_csv_atomic(frame, path):
    pending = path.with_suffix('.csv.partial')
    with pending.open('wb') as output:
        for start in range(0, max(1, len(frame)), 2_000):
            block = frame.iloc[start:start+2_000].to_csv(index=False, header=start == 0).encode('utf-8')
            output.flush()
            reserve_packed(len(block))
            output.write(block)
    pending.replace(path)

if manifest_reuse:
    manifests = {name: pd.read_csv(path, dtype={'stem': str, 'scene': str, 'obj': str, 'part': str},
                   keep_default_na=False) for name, path in manifest_paths.items()}
else:
    by_stem = packed_annotations.set_index('stem', drop=False)
    manifests, no_separator = {}, Counter()
    for split in ('seen', 'unseen'):
        stems = [stem for stem in sorted(official_test_stems) if split_parts(stem)[3] in test_keys[split]]
        frame = by_stem.loc[stems].copy()
        frame['split'] = split
        manifests['test_'+split] = frame[COLUMNS].reset_index(drop=True)
        no_separator['test_'+split] = sum(not extract_obj_name(prompt)[1] for prompt in frame['prompt'])
    UNSEEN_NAMES = set(manifests['test_unseen']['obj_name'])
    candidates = by_stem.loc[sorted(input_stems)].copy()
    mask_a = candidates['scene'].isin(ALL_TEST_SCENES)
    after_a = candidates.loc[~mask_a]
    no_separator['input_after_scene_exclusion'] = sum(not extract_obj_name(prompt)[1] for prompt in after_a['prompt'])
    mask_b = after_a['obj_name'].isin(UNSEEN_NAMES)
    removed_names = Counter(after_a.loc[mask_b, 'obj_name'])
    eligible = after_a.loc[~mask_b].copy()
    eligible['split'] = 'train'
    scenes = np.array(sorted(set(eligible['scene'])), dtype=object)
    if len(scenes) < 2:
        raise RuntimeError('Need at least two eligible scenes for disjoint train/validation manifests.')
    n_val = min(int(np.ceil(0.02*len(scenes))), len(scenes)-1)
    val_scenes = set(np.random.default_rng(0).choice(scenes, size=n_val, replace=False))
    in_val = eligible['scene'].isin(val_scenes)
    manifests['train'] = eligible.loc[~in_val, COLUMNS].reset_index(drop=True)
    manifests['val'] = eligible.loc[in_val, COLUMNS].reset_index(drop=True)
    manifests['val']['split'] = 'val'
    assert not (set(manifests['train']['scene']) & set(manifests['val']['scene']))
    for name in ('train', 'val'):
        assert not (set(manifests[name]['scene']) & ALL_TEST_SCENES)
        assert not (set(manifests[name]['obj_name']) & UNSEEN_NAMES)
    for name, frame in manifests.items():
        save_csv_atomic(frame, manifest_paths[name])
    manifest_meta = {'key': manifest_key, 'counts': {name: len(frame) for name, frame in manifests.items()},
        'input_total_pairs': len(input_stems), 'input_unpaired_prompts': ann_meta['input_unpaired_prompts'],
        'input_unpaired_labels': ann_meta['input_unpaired_labels'], 'removed_test_scene': int(mask_a.sum()),
        'removed_unseen_name_after_a': int(mask_b.sum()), 'removed_names_top20': removed_names.most_common(20),
        'eligible_before_val': len(eligible), 'eligible_scenes_before_val': len(scenes),
        'train_scenes': int(manifests['train']['scene'].nunique()), 'val_scenes': len(val_scenes),
        'val_scene_fraction': len(val_scenes)/len(scenes), 'no_separator': dict(no_separator),
        'unseen_names': sorted(UNSEEN_NAMES),
        'csv_sha256': {name: file_sha(path) for name, path in manifest_paths.items()}}
    packed_atomic_json(manifest_meta_path, manifest_meta)
UNSEEN_NAMES = set(manifest_meta['unseen_names'])
stage_done('manifest_build_s', manifest_started, manifest_reuse)
print('Input pairs:', manifest_meta['input_total_pairs'], '| removed (a):', manifest_meta['removed_test_scene'],
      '| removed (b) after (a):', manifest_meta['removed_unseen_name_after_a'],
      '| final train:', len(manifests['train']), '| train scenes:', manifest_meta['train_scenes'],
      '| val:', len(manifests['val']), '| val scenes:', manifest_meta['val_scenes'])
print('20 most frequent removed names for (b):', manifest_meta['removed_names_top20'])
print('No-separator prompt counts:', manifest_meta['no_separator'])
examples = pd.concat([frame[['prompt', 'obj_name']] for frame in manifests.values()], ignore_index=True)
print('30 reproducible random prompt -> obj_name examples:')
for row in examples.sample(n=min(30, len(examples)), random_state=0).itertuples(index=False):
    print(f'  {row.prompt!r} -> {row.obj_name!r}')



all_manifest_scenes = set().union(*(set(frame['scene']) for frame in manifests.values()))
if not all_manifest_scenes <= set(image_table['scene']):
    raise DataIntegrityError('Some manifest scenes are missing from the packed image blob.')
module_started = time.perf_counter()
PACKED_IO_SOURCE = r'''"""Portable access to ga_eval/packed; cached metadata, memmapped labels, cached JPEG handles.

root is ga_eval/ (or its packed/ directory). Metadata contains no absolute paths.
Treat returned DataFrames and the labels memmap as read-only. read_jpeg returns a
fully decoded RGB PIL.Image, independent of its compressed input buffer.
"""
import io
import os
import threading
import zlib
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

_local = threading.local()

def _directory(root):
    path = Path(os.path.abspath(os.fspath(root)))
    return path if path.name == 'packed' else path / 'packed'

@lru_cache(maxsize=8)
def _open(directory, pid):
    path = Path(directory)
    annotations = pd.read_parquet(path / 'annotations.parquet')
    labels = np.load(path / 'labels.npy', mmap_mode='r', allow_pickle=False)
    images = pd.read_parquet(path / 'images.parquet')
    if labels.dtype != np.float32 or labels.ndim != 2 or labels.shape[1] != 6:
        raise ValueError('Expected float32 labels.npy with shape (total_rows,6)')
    if annotations['stem'].duplicated().any() or images['scene'].duplicated().any():
        raise ValueError('Packed annotation stems and image scenes must be unique')
    starts = annotations['label_start'].to_numpy(dtype=np.int64)
    counts = annotations['label_count'].to_numpy(dtype=np.int64)
    if np.any(starts < 0) or np.any(counts < 0) or np.any(starts+counts > len(labels)):
        raise ValueError('Annotation label slices exceed labels.npy')
    index = {row.scene: {'offset': int(row.offset), 'nbytes': int(row.nbytes), 'crc32': int(row.crc32)}
             for row in images.itertuples(index=False)}
    return annotations, labels, index, path / 'images.bin'

def open_packed(root):
    """Return (annotations DataFrame, read-only labels memmap, image index dict, blob Path).

    Labels for an annotation row are labels[row.label_start:row.label_start+row.label_count].
    The PID cache key prevents inherited memmaps/metadata from being reused after fork.
    """
    return _open(str(_directory(root)), os.getpid())

def _handle(path):
    pid = os.getpid()
    if getattr(_local, 'pid', None) != pid:
        for handle in getattr(_local, 'handles', {}).values():
            handle.close()
        _local.pid, _local.handles = pid, {}
    key = str(path)
    handle = _local.handles.get(key)
    if handle is None or handle.closed:
        handle = open(path, 'rb')
        _local.handles[key] = handle
    return handle

def read_jpeg(root, scene):
    """Seek/read one CRC-checked JPEG using a cached handle per process and thread.

    Independent handles avoid seek races between threads and shared file offsets
    between forked training DataLoader workers. No source image file is opened.
    """
    annotations, labels, index, path = open_packed(root)
    record = index[str(scene)]
    handle = _handle(path)
    handle.seek(record['offset'])
    data = handle.read(record['nbytes'])
    if len(data) != record['nbytes'] or (zlib.crc32(data) & 0xffffffff) != record['crc32']:
        raise IOError(f'Packed JPEG is truncated or has a CRC mismatch: {scene}')
    with Image.open(io.BytesIO(data)) as image:
        image.load()
        return image.convert('RGB')

def close_packed(root=None):
    """Close this thread's cached JPEG handles and cached label maps before rebuilding."""
    directory = _directory(root) if root is not None else None
    for key, handle in list(getattr(_local, 'handles', {}).items()):
        if directory is None or Path(key).parent == directory:
            handle.close()
            del _local.handles[key]
    # Cache clearing releases metadata. Existing memmap references remain valid;
    # a notebook rebuilding files must first drop its references (cell 4 does).
    _open.cache_clear()
'''
helper_path = OUT / 'packed_io.py'
if not helper_path.is_file() or helper_path.read_text(encoding='utf-8') != PACKED_IO_SOURCE:
    reserve_packed(len(PACKED_IO_SOURCE.encode('utf-8')))
    helper_path.write_text(PACKED_IO_SOURCE, encoding='utf-8')
spec = importlib.util.spec_from_file_location('packed_io', helper_path)
packed_io = importlib.util.module_from_spec(spec)
sys.modules['packed_io'] = packed_io
spec.loader.exec_module(packed_io)
packed_annotations, packed_labels, packed_images, packed_blob_path = packed_io.open_packed(OUT)
stage_done('helper_import_s', module_started)
stage_timings['cell4_total_s'] = time.perf_counter()-cell4_started
packed_sizes = {name: (PACKED / name).stat().st_size for name in
                ('labels.npy', 'annotations.parquet', 'images.bin', 'images.parquet')}
packed_atomic_json(OUT / 'stage_timings.json', {'seconds': stage_timings, 'reused': stage_reused})
packed_atomic_json(PACKED / 'stream_complete.json', {
    'source_version': SOURCE_VERSION, 'split_id': SPLIT_ID, 'image_key': IMAGE_KEY, 'annotation_key': ANN_KEY,
    'files': packed_sizes, 'manifest_key': manifest_meta['key']})
# Delete the two LOCAL source ZIPs only after every stage and its completion marker succeeded.
for kind in ('grasp_instructions', 'grasp_label_positive'):
    path = TMP / (kind+'.zip')
    if path.is_file():
        path.unlink()
        print('Deleted completed source ZIP:', path.name)
print('Packed file sizes (bytes):', packed_sizes)
print(f'Cell 4 complete: {len(SUBSET_SCENES):,} subset scenes, {len(packed_annotations):,} stems, '
      f'labels={packed_labels.shape}, images={len(packed_images):,}; total={stage_timings["cell4_total_s"]:.2f}s.')
print('Timing targets: image stream <180s; local annotation pass <600s (actual timings printed above).')
reserve_packed()


## Cell 5 — Write and import eval_utils.py

Reusable GT conversion, image-coordinate polygons, symmetric angle difference, strict success, success rate and harmonic mean.


In [ ]:
# CELL 5 — Write AND import the reusable evaluator (CPU; no training).
EVAL_UTILS_SOURCE = r'''"""CPU grasp geometry in LGD's image-coordinate convention; no model or training code."""
from math import pi
import numpy as np
import torch
from shapely.geometry import Polygon

def load_gt_array(raw, scale=1.0):
    """Convert a raw float32 (N,6) array/view to [cx,cy,length,width,angle_rad].

    Column 0 is ignored. Source row (_,x,y,w,h,theta_deg) maps to center
    (row=y,col=x), angle=-theta_deg*pi/180, length=w, width=h. Scale affects
    x,y,w,h only. Input memmap views are never mutated.
    """
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError('scale must be finite and positive')
    raw = np.asarray(raw)
    if raw.ndim != 2 or raw.shape[1] != 6:
        raise ValueError('Expected raw array with shape (N,6)')
    if raw.dtype != np.float32:
        raise TypeError('Expected float32 raw labels')
    result = raw[:, 1:6].astype(np.float64, copy=True)
    result[:, :4] *= scale
    result[:, 4] = -np.deg2rad(raw[:, 5].astype(np.float64))
    if not np.isfinite(result).all() or np.any(result[:, 2:4] <= 0):
        raise ValueError('GT has nonfinite values or nonpositive dimensions')
    return result

def load_gt(path, scale=1.0):
    """Compatibility wrapper: load one CPU tensor file, then delegate to load_gt_array."""
    tensor = torch.load(path, map_location='cpu', weights_only=True)
    if not isinstance(tensor, torch.Tensor):
        raise TypeError(f'Expected a torch tensor: {path}')
    return load_gt_array(tensor.detach().cpu().numpy(), scale=scale)

def rect_polygon(cx, cy, length, width, angle):
    """Four-corner Polygon in (x=column,y=row) coordinates, with image y pointing down.

    Positive angle rotates counterclockwise on the displayed image, matching LGD.
    The length-axis vector is (cos(angle), -sin(angle)); the width-axis vector is
    (sin(angle), cos(angle)). Thus a stored positive theta_deg becomes a negative
    angle and its length axis slopes down to the right. Polygons are not clipped.
    """
    values = np.asarray([cx, cy, length, width, angle], dtype=float)
    if not np.isfinite(values).all() or length <= 0 or width <= 0:
        raise ValueError('Rectangle parameters must be finite with positive length/width')
    u = np.array([np.cos(angle), -np.sin(angle)]) * (length/2)
    v = np.array([np.sin(angle), np.cos(angle)]) * (width/2)
    center = np.array([cx, cy])
    return Polygon([center-u-v, center+u-v, center+u+v, center-u+v])

def angle_diff(a, b):
    """Symmetric angular distance modulo pi (parallel/opposite axes are equivalent)."""
    return abs(((a-b+pi/2) % pi)-pi/2)

def polygon_iou(a, b):
    """Continuous, unclipped polygon IoU, without the separate angle gate."""
    first, second = rect_polygon(*a), rect_polygon(*b)
    intersection = first.intersection(second).area
    union = first.area+second.area-intersection
    return float(intersection/union) if union > 0 else 0.0

def is_success(pred, gts, iou_thr=0.25, ang_thr=pi/6):
    """Strict gates: ANY GT with angle_diff < ang_thr AND IoU > iou_thr succeeds."""
    pred, gts = np.asarray(pred, dtype=float), np.asarray(gts, dtype=float)
    if pred.shape != (5,) or gts.ndim != 2 or gts.shape[1] != 5:
        raise ValueError('Expected pred shape (5,) and gts shape (N,5)')
    if not np.isfinite(pred).all() or np.any(pred[2:4] <= 0):
        return False
    if not np.isfinite(gts).all() or np.any(gts[:, 2:4] <= 0):
        raise ValueError('GT rectangles must be finite with positive length/width')
    for gt in gts:
        if angle_diff(pred[4], gt[4]) < ang_thr and polygon_iou(pred, gt) > iou_thr:
            return True
    return False

def success_rate(pairs):
    """Fraction of successful (prediction, GT-array) pairs; an empty input is an error."""
    count, successes = 0, 0
    for prediction, gts in pairs:
        successes += int(is_success(prediction, gts))
        count += 1
    if count == 0:
        raise ValueError('success_rate requires at least one pair')
    return successes/count

def harmonic_mean(s, u):
    """2*s*u/(s+u), or 0 when both rates are zero."""
    if not (np.isfinite(s) and np.isfinite(u) and 0 <= s <= 1 and 0 <= u <= 1):
        raise ValueError('s and u must be finite success rates in [0,1]')
    return 2*s*u/(s+u) if s+u else 0.0
'''
utility_path = OUT / 'eval_utils.py'
if utility_path.is_file() and utility_path.read_text(encoding='utf-8') == EVAL_UTILS_SOURCE:
    print('Skip eval_utils.py write: contents already match.')
else:
    reserve_output(len(EVAL_UTILS_SOURCE.encode('utf-8')))
    utility_path.write_text(EVAL_UTILS_SOURCE, encoding='utf-8')
spec = importlib.util.spec_from_file_location('eval_utils', utility_path)
eval_utils = importlib.util.module_from_spec(spec)
sys.modules['eval_utils'] = eval_utils
spec.loader.exec_module(eval_utils)
from eval_utils import load_gt, load_gt_array, rect_polygon, angle_diff, is_success, success_rate, harmonic_mean, polygon_iou
UTILITY_SHA = file_sha(utility_path)
print('Cell 5 complete:', utility_path, '| imported functions:',
      'load_gt, load_gt_array, rect_polygon, angle_diff, is_success, success_rate, harmonic_mean; + polygon_iou')
print('Axis=(cos(angle), -sin(angle)); theta_deg sign is flipped; strict IoU>0.25 and angle<30 degrees.')


## Cell 6 — G1 assertions and packed label statistics

Run the same G1 checks from labels.npy and annotations.parquet. Train statistics merge adjacent selected label slices into bounded vectorized reads; no per-sample file access.


In [ ]:
# CELL 6 — G1 assertions and train-label statistics. Failures stay visible in the final REPORT.
cell6_started = time.perf_counter()
packed_annotations, packed_labels, packed_images, packed_blob_path = packed_io.open_packed(OUT)
annotation_slices = {row.stem: (int(row.label_start), int(row.label_count))
                     for row in packed_annotations.itertuples(index=False)}
def packed_raw(stem):
    start, count = annotation_slices[stem]
    return packed_labels[start:start+count]
def packed_gts(stem):
    return load_gt_array(packed_raw(stem))
G1_KEY = digest_json([manifest_meta['key'], ann_meta['labels_payload_sha256'], UTILITY_SHA, 1000, 500, 0, 'g1-packed-v1'])
g1_path = OUT / 'g1.json'
reuse = g1_path.is_file() and read_json(g1_path)['key'] == G1_KEY
if reuse:
    g1 = read_json(g1_path)
    print('Skip G1 recomputation: manifests, utility source and sampling configuration match.')
else:
    from skimage.draw import polygon as raster_polygon
    from shapely.geometry import box
    rng = np.random.default_rng(0)
    tests = pd.concat([manifests['test_seen'], manifests['test_unseen']], ignore_index=True)
    nonempty = tests.loc[tests['label_count'].astype(int) > 0].reset_index(drop=True)
    if nonempty.empty:
        raise RuntimeError('No nonempty test labels are available for G1.')
    replacement = len(nonempty) < 1000
    sample = nonempty.iloc[rng.choice(len(nonempty), size=1000, replace=replacement)]
    first_gts = [packed_gts(row.stem)[0].copy() for row in sample.itertuples(index=False)]
    checks = []
    def check(name, condition, detail):
        # Explicitly record failed assertions so cells 7 and 8 can still produce diagnostics.
        try:
            assert bool(condition), detail
        except AssertionError:
            status = 'FAIL'
            warnings.warn(f'G1 FAIL — {name}: {detail}')
        else:
            status = 'PASS'
        checks.append({'check': name, 'status': status, 'detail': detail})
    identity_rate = success_rate((gt, gt[None, :]) for gt in first_gts)
    # Also test each first prediction against its actual FULL label array, not just itself.
    actual_rate = success_rate((gt, packed_gts(row.stem)) for gt, row in
                              zip(first_gts, sample.itertuples(index=False)))
    check('1 GT-as-prediction', identity_rate == actual_rate == 1.0,
          f'n=1000, rate={actual_rate:.3f}; empty test files excluded={len(tests)-len(nonempty)}; replacement={replacement}')
    failures_31, successes_180, shifted_ious = [], [], []
    for gt in first_gts:
        rotated = gt.copy()
        rotated[4] += np.deg2rad(31)
        failures_31.append(not is_success(rotated, gt[None, :]))
        rotated[4] = gt[4]+np.pi
        successes_180.append(is_success(rotated, gt[None, :]))
        shifted = gt.copy()
        shifted[:2] += gt[2]*np.array([np.cos(gt[4]), -np.sin(gt[4])])
        shifted_ious.append(polygon_iou(shifted, gt))
    check('2a +31 degrees', all(failures_31), f'{sum(failures_31)}/1000 fail against that single GT')
    check('2b +180 degrees', all(successes_180), f'{sum(successes_180)}/1000 succeed against that single GT')
    check('3 Shift by length', max(shifted_ious) < 0.25, f'max IoU={max(shifted_ious):.8g}; n=1000')
    scene_groups = {scene: group for scene, group in nonempty.groupby('scene')
                    if group['label_count'].astype(int).sum() >= 2}
    scene_names = sorted(scene_groups)
    if not scene_names:
        raise RuntimeError('No test scene has at least two GT rows for the raster comparison.')
    @lru_cache(maxsize=64)
    def scene_gts(scene):
        return np.concatenate([packed_gts(stem) for stem in scene_groups[scene]['stem']], axis=0)
    def raster_iou(first, second):
        masks = []
        for value in (first, second):
            points = np.asarray(rect_polygon(*value).exterior.coords)[:4]
            rr, cc = raster_polygon(points[:, 1], points[:, 0], shape=(416, 416))
            mask = np.zeros((416, 416), dtype=bool)
            mask[rr, cc] = True
            masks.append(mask)
        union = np.count_nonzero(masks[0] | masks[1])
        return float(np.count_nonzero(masks[0] & masks[1])/union) if union else 0.0
    diffs, clipped_diffs, outside_pairs = [], [], 0
    canvas = box(0, 0, 416, 416)
    for scene in tqdm(rng.choice(scene_names, size=500, replace=True), desc='Raster IoU G1', unit='pairs'):
        pool = scene_gts(scene)
        i, j = rng.choice(len(pool), size=2, replace=False)
        first, second = pool[i], pool[j]
        analytic, raster = polygon_iou(first, second), raster_iou(first, second)
        diffs.append(abs(analytic-raster))
        a, b = rect_polygon(*first), rect_polygon(*second)
        outside_pairs += int(not (canvas.covers(a) and canvas.covers(b)))
        a, b = a.intersection(canvas), b.intersection(canvas)
        intersection = a.intersection(b).area
        union = a.area+b.area-intersection
        clipped_diffs.append(abs((intersection/union if union else 0)-raster))
    mean_diff = float(np.mean(diffs))
    check('4 Continuous vs raster IoU', mean_diff < 0.02,
          f'n=500 same-scene distinct-row pairs; mean |diff|={mean_diff:.6f}; '
          f'max={max(diffs):.6f}; out-of-canvas pairs={outside_pairs}; '
          f'clip-matched diagnostic mean={np.mean(clipped_diffs):.6f}')
    # The required comparison above uses UNCLIPPED continuous IoU. The separately
    # reported clipped diagnostic identifies frame clipping; it never changes the evaluator.
# Vectorized train statistics from contiguous runs of the memory-mapped label array.
    train_ranges = packed_annotations.set_index('stem').loc[manifests['train']['stem'], ['label_start', 'label_count']]
    ns = train_ranges['label_count'].to_numpy(dtype=np.int64)
    starts = train_ranges['label_start'].to_numpy(dtype=np.int64)
    nonzero = ns > 0
    order = np.argsort(starts[nonzero], kind='stable')
    begins = starts[nonzero][order]
    ends = begins+ns[nonzero][order]
    if not len(ns) or not len(begins):
        raise RuntimeError('Training manifest has no grasp rows to summarize.')
    # Merge adjacent retained slices; stream the result in <=1M-row RAM batches.
    cuts = np.r_[True, begins[1:] != ends[:-1]]
    group_starts = np.flatnonzero(cuts)
    intervals = list(zip(begins[group_starts], ends[np.r_[group_starts[1:]-1, len(ends)-1]]))
    total_rows, outside_rows, invalid_rows = 0, 0, 0
    lo, hi = np.full(3, np.inf), np.full(3, -np.inf)
    with tqdm(total=int(ns.sum()), desc='Packed train label stats', unit='rows') as progress:
        for begin, end in intervals:
            for position in range(int(begin), int(end), 1_000_000):
                raw = packed_labels[position:min(position+1_000_000, int(end))]
                values = raw[:, [5, 3, 4]]
                finite = np.isfinite(raw[:, 1:6]).all(axis=1)
                valid = finite & (raw[:, 3] > 0) & (raw[:, 4] > 0)
                invalid_rows += int((~valid).sum())
                if not finite.all():
                    raise ValueError(f'Nonfinite packed train labels at row interval {position}:{position+len(raw)}.')
                lo, hi = np.minimum(lo, values.min(axis=0)), np.maximum(hi, values.max(axis=0))
                outside_rows += int(((raw[:, 1] < 0) | (raw[:, 1] >= 416) |
                                     (raw[:, 2] < 0) | (raw[:, 2] >= 416)).sum())
                total_rows += len(raw)
                progress.update(len(raw))
    assert total_rows == int(ns.sum()), 'Packed train intervals lost or duplicated grasp rows.'
    train_stats = {'files': len(ns), 'rows': total_rows,
        'N_min_median_max': [int(ns.min()), float(np.median(ns)), int(ns.max())],
        'theta_deg_range': [float(lo[0]), float(hi[0])], 'w_range': [float(lo[1]), float(hi[1])],
        'h_range': [float(lo[2]), float(hi[2])], 'outside_xy_rows': outside_rows,
        'outside_xy_fraction': outside_rows/total_rows, 'nonpositive_or_invalid_rows': invalid_rows}
    check('5 Train label stats', invalid_rows == 0,
          f'files={len(ns):,}; rows={total_rows:,}; invalid/nonpositive rows={invalid_rows:,}; '
          f'outside-center fraction={outside_rows/total_rows:.6%}; packed runs={len(intervals):,}')

    g1 = {'key': G1_KEY, 'checks': checks, 'train_label_stats': train_stats,
          'raster_mean_abs_diff': mean_diff, 'raster_outside_pairs': outside_pairs,
          'raster_clipped_mean_abs_diff': float(np.mean(clipped_diffs))}
    packed_atomic_json(g1_path, g1)
print(pd.DataFrame(g1['checks']).to_string(index=False))
print('Train label statistics:', json.dumps(g1['train_label_stats'], indent=2))
print('Cell 6 complete:', sum(row['status'] == 'PASS' for row in g1['checks']), '/', len(g1['checks']),
      'G1 assertions passed. Any FAIL is raised after the final REPORT is printed.')

stage_done('g1_s', cell6_started, reuse)
packed_atomic_json(OUT / 'stage_timings.json', {'seconds': stage_timings, 'reused': stage_reused})


## Cell 7 — Visual check

Save/display a seeded 3×4 test grid with all GT rectangles in green and the first grasp axis in red.


In [ ]:
# CELL 7 — 3x4 visual check: all GTs green; first GT length axis red.
cell7_started = time.perf_counter()
from IPython.display import display, Image as DisplayImage
png_path, png_meta_path = OUT / 'gt_check.png', OUT / 'gt_check.meta.json'
visual_key = digest_json([manifest_meta['key'], ann_meta['labels_payload_sha256'], image_meta['images_sha256'], UTILITY_SHA, 'grid-3x4-packed-v1'])
reuse = png_path.is_file() and png_meta_path.is_file()
if reuse:
    visual_meta = read_json(png_meta_path)
    reuse = visual_meta['key'] == visual_key and visual_meta['png_sha256'] == file_sha(png_path)
if reuse:
    print('Skip figure rendering: cached 3x4 grid matches these manifests and evaluator.')
else:
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection
    from PIL import Image
    import textwrap
    samples = pd.concat([manifests['test_seen'], manifests['test_unseen']], ignore_index=True)
    samples = samples.loc[samples['label_count'].astype(int) > 0]
    if samples.empty:
        raise RuntimeError('No nonempty test samples for visual inspection.')
    chosen = samples.sample(n=12, replace=len(samples) < 12, random_state=17)
    figure, axes = plt.subplots(3, 4, figsize=(17, 14), layout='constrained')
    for ax, row in zip(axes.flat, chosen.itertuples(index=False)):
        with packed_io.read_jpeg(OUT, row.scene) as image:
            if image.size != (416, 416):
                raise ValueError(f'Expected 416x416 image: {row.scene}; got {image.size}')
            ax.imshow(np.asarray(image.convert('RGB')))
        gts = load_gt_array(packed_labels[int(row.label_start):int(row.label_start)+int(row.label_count)])
        rectangles = [np.asarray(rect_polygon(*gt).exterior.coords) for gt in gts]
        ax.add_collection(LineCollection(rectangles, colors='lime', linewidths=0.6, alpha=0.7))
        cx, cy, length, width, angle = gts[0]
        delta = length/2*np.array([np.cos(angle), -np.sin(angle)])
        endpoints = np.array([cx, cy])+np.array([-delta, delta])
        ax.plot(endpoints[:, 0], endpoints[:, 1], color='red', linewidth=2)
        ax.set_title(textwrap.fill(row.prompt, width=44), fontsize=9)
        ax.set_xlim(-0.5, 415.5)
        ax.set_ylim(415.5, -0.5)
        ax.axis('off')
    pending_png = OUT / 'gt_check.partial.png'
    reserve_packed(15_000_000)
    figure.savefig(pending_png, dpi=120, format='png')
    plt.close(figure)
    reserve_packed()
    pending_png.replace(png_path)
    visual_meta = {'key': visual_key, 'png_sha256': file_sha(png_path), 'stems': chosen['stem'].tolist()}
    packed_atomic_json(png_meta_path, visual_meta)
display(DisplayImage(filename=str(png_path)))
print(f'Cell 7 complete: saved/displayed {png_path}; 12 test samples; green=all GTs, red=first grasp axis.')

stage_done('visual_s', cell7_started, reuse)
packed_atomic_json(OUT / 'stage_timings.json', {'seconds': stage_timings, 'reused': stage_reused})


## Cell 8 — REPORT

One copy-paste block with split/image/manifest counts, exclusions, G1 checks, label statistics and LGD commit.


In [ ]:
# CELL 8 — Compact, copy-paste REPORT. No model training or claimed model evaluation scores.
reserve_output()
report = ['REPORT', f'Notebook: ga-pp-testset-eval; LGD HEAD: {LGD_COMMIT}']
report.append('Official splits: ' + '; '.join(
    f'{name}={len(test_keys[name])} keys/{len(test_scenes[name])} scenes' for name in ('seen', 'unseen'))
    + f'; union={len(ALL_TEST_SCENES)} scenes')
report.append('Test images: ' + json.dumps(image_fetch, separators=(',', ':')))
report.append('Test annotations: ' + '; '.join(f'{name}={annotation_counts[name]["pairs"]} pairs, '
    f'{annotation_counts[name]["keys_with_zero_pairs"]} zero-pair keys' for name in ('seen', 'unseen')))
report.append('Manifest pairs: ' + '; '.join(f'{name}={len(frame):,}' for name, frame in manifests.items()))
report.append(f'Input pairs={manifest_meta["input_total_pairs"]:,}; removed (a) test scenes='
    f'{manifest_meta["removed_test_scene"]:,}; removed (b) unseen names after (a)='
    f'{manifest_meta["removed_unseen_name_after_a"]:,}; eligible before val={manifest_meta["eligible_before_val"]:,}')
report.append(f'Final train scenes={manifest_meta["train_scenes"]:,}; val scenes={manifest_meta["val_scenes"]:,}; '
    f'val scene fraction={manifest_meta["val_scene_fraction"]:.3%}; seed=0; UNSEEN_NAMES={len(UNSEEN_NAMES)}')
report.append('Top20 removed names (b): ' + json.dumps(manifest_meta['removed_names_top20'], separators=(',', ':')))
report.append('No-separator counts: ' + json.dumps(manifest_meta['no_separator'], separators=(',', ':')))
report.append('G1:')
for row in g1['checks']:
    report.append(f'  {row["status"]:4s} | {row["check"]} | {row["detail"]}')
report.append('Train label stats: ' + json.dumps(g1['train_label_stats'], separators=(',', ':')))
report.append(f'Output size before REPORT: {directory_bytes(OUT):,} bytes (<6,000,000,000); visual=gt_check.png')
report.append('Packed file sizes (bytes): ' + json.dumps(packed_sizes, separators=(',', ':')))
report.append('Stage timings (seconds): ' + json.dumps({k: round(v, 3) for k, v in stage_timings.items()}, separators=(',', ':')))
report.append('Reused stages: ' + json.dumps(stage_reused, separators=(',', ':')))
report.append('END REPORT')
report_text = '\n'.join(report)
reserve_output(len(report_text.encode('utf-8')))
(OUT / 'REPORT.txt').write_text(report_text+'\n', encoding='utf-8')
print(report_text)
assert all(row['status'] == 'PASS' for row in g1['checks']), 'G1 has FAIL rows; diagnostics are preserved in REPORT.txt.'
